# Диагностика и фиксация auto-dismiss кастомной клавиатуры

Этот notebook фиксирует практический workflow для поиска причины автоскрытия клавиатуры в React-клиенте и верификации исправления.

## 1. Create Minimal React Keypad Sandbox

Цель: собрать минимальный пример, повторяющий поведение из App.jsx: активное поле факта, видимость клавиатуры, открытие/закрытие при focus/blur, ввод через кнопки.

In [ ]:
import { useRef, useState } from 'react';

export function KeypadSandbox() {
  const [activeFactCode, setActiveFactCode] = useState('item-1');
  const [value, setValue] = useState('');
  const [keypadVisible, setKeypadVisible] = useState(true);
  const keypadRef = useRef(null);

  function handleFocus() {
    setActiveFactCode('item-1');
    setKeypadVisible(true);
  }

  function handleBlur() {
    setActiveFactCode('');
    setKeypadVisible(false);
  }

  function append(char) {
    setValue(prev => prev + char);
  }

  return (
    <div style={{ maxWidth: 380, margin: '16px auto', padding: 12 }}>
      <input
        value={value}
        onFocus={handleFocus}
        onBlur={handleBlur}
        readOnly
        placeholder="Фактическое количество"
        style={{ width: '100%', padding: 8 }}
      />
      {keypadVisible && activeFactCode ? (
        <div ref={keypadRef} style={{ marginTop: 10, border: '1px solid #ccc', padding: 10 }}>
          <div style={{ display: 'grid', gridTemplateColumns: 'repeat(3, 1fr)', gap: 8 }}>
            {['1', '2', '3', '4', '5', '6', '7', '8', '9', '+', '0', '⌫'].map(key => (
              <button key={key} type="button" onClick={() => append(key === '⌫' ? '' : key)}>
                {key}
              </button>
            ))}
          </div>
        </div>
      ) : null}
    </div>
  );
}

## 2. Instrument focus/blur/pointer Event Timeline

Цель: добавить сбор таймлайна событий pointer и focus/blur, чтобы увидеть точный порядок, приводящий к скрытию клавиатуры.

События для логирования:
- pointerdown
- pointerup
- pointercancel
- focus
- blur

Для каждого события фиксируйте: `type`, `target`, `timeStamp`, текущие `activeFactCode` и `keypadVisible`.

In [ ]:
function createEventLogger(limit = 120) {
  const timeline = [];

  function push(eventName, meta = {}) {
    const row = {
      event: eventName,
      t: performance.now().toFixed(2),
      ...meta
    };
    timeline.push(row);
    if (timeline.length > limit) timeline.shift();
    console.table([row]);
  }

  function dump() {
    console.table(timeline);
    return timeline;
  }

  function clear() {
    timeline.length = 0;
  }

  return { push, dump, clear };
}

const logger = createEventLogger();

## 3. Reproduce Rapid-Tap and Between-Keys Failure Cases

Шаги для воспроизведения:
1. Сфокусировать поле факта.
2. Быстро нажимать `1`, `2`, `3`, `4` подряд.
3. Повторить, намеренно попадая в промежутки между кнопками.

Ожидаемая ошибка в дефектной версии:
- после одного из `blur` состояние `activeFactCode` очищается;
- `keypadVisible` становится `false`.

Проверьте это через `logger.dump()` и найдите событие, где `blur` приходит сразу после `pointerdown` в области клавиатуры.

In [ ]:
// Мини-утилита для ручного прогона в браузерной песочнице
function attachDiagnostics({ inputEl, keypadEl, getState }) {
  const events = ['pointerdown', 'pointerup', 'pointercancel', 'focus', 'blur'];

  function onAny(event) {
    const state = typeof getState === 'function' ? getState() : {};
    logger.push(event.type, {
      target: event.target?.className || event.target?.tagName || 'unknown',
      activeFactCode: state.activeFactCode || '',
      keypadVisible: String(Boolean(state.keypadVisible))
    });
  }

  for (const type of events) {
    inputEl?.addEventListener(type, onAny, true);
    keypadEl?.addEventListener(type, onAny, true);
    document.addEventListener(type, onAny, true);
  }

  return () => {
    for (const type of events) {
      inputEl?.removeEventListener(type, onAny, true);
      keypadEl?.removeEventListener(type, onAny, true);
      document.removeEventListener(type, onAny, true);
    }
  };
}

## 4. Implement Stable Keypad Interaction Guard

Добавьте mutable-флаг взаимодействия внутри клавиатуры, например `keypadInteractionRef`.

Правило:
- при `blur` поля факта не закрывать клавиатуру, если флаг активен;
- сбрасывать флаг после завершения pointer-взаимодействия.

Этим устраняется ложное закрытие при скоростных тачах и микросмещении пальца.

In [ ]:
// Фрагмент исправления для React-компонента
const keypadInteractionRef = React.useRef(false);

function handleFactBlur() {
  if (keypadInteractionRef.current) return;
  setActiveFactCode('');
  setKeypadVisible(false);
}

function markKeypadInteraction() {
  keypadInteractionRef.current = true;
}

function clearKeypadInteraction() {
  setTimeout(() => {
    keypadInteractionRef.current = false;
  }, 0);
}

## 5. Handle Container-Level Pointer Capture Events

Ключевая идея: слушать pointer capture на контейнере клавиатуры, а не только на кнопках.

Почему это важно:
- при попадании пальца между кнопками событие всё равно происходит в контейнере;
- флаг взаимодействия остаётся корректным, и `blur` не приводит к скрытию панели.

In [ ]:
// JSX контейнера клавиатуры
<div
  ref={keypadRef}
  className="fact-keypad"
  onPointerDownCapture={markKeypadInteraction}
  onPointerUpCapture={clearKeypadInteraction}
  onPointerCancelCapture={clearKeypadInteraction}
>
  <div className="fact-keypad-grid">
    <button type="button" onPointerDown={e => e.preventDefault()} onClick={() => appendToActiveFact('1')}>1</button>
    <button type="button" onPointerDown={e => e.preventDefault()} onClick={() => appendToActiveFact('2')}>2</button>
    <button type="button" onPointerDown={e => e.preventDefault()} onClick={() => appendToActiveFact('3')}>3</button>
    {/* ... остальные кнопки ... */}
  </div>
</div>

## 6. Refine Close Conditions for Intentional Dismiss Only

Разрешённые сценарии закрытия:
- явный tap вне контекста input/keypad;
- смена режима (переход на другой экран, закрытие модала, завершение действия);
- явная команда закрытия.

Неразрешённые сценарии закрытия:
- быстрый повторный тап по кнопкам;
- тап в промежуток между кнопками внутри области клавиатуры.

In [ ]:
// Псевдокод ожидаемой логики закрытия
function maybeCloseKeypad({ reason, withinKeypadInteraction }) {
  const allowedReasons = new Set(['outside_tap', 'mode_switch', 'explicit_close']);
  if (!allowedReasons.has(reason)) return false;
  if (withinKeypadInteraction) return false;
  return true;
}

console.log(maybeCloseKeypad({ reason: 'outside_tap', withinKeypadInteraction: false })); // true
console.log(maybeCloseKeypad({ reason: 'blur', withinKeypadInteraction: true })); // false

## 7. Add Regression Tests for Input, Delete, Save, and Scroll Behavior

Покройте автотестами:
- ввод выражения `3+7+12`;
- удаление символа кнопкой `⌫`;
- сохранение/подтверждение значения;
- прокрутку к активной карточке, когда клавиатура открыта;
- отсутствие скрытия клавиатуры при rapid taps и gap taps.

Рекомендуемые инструменты:
- React Testing Library для компонентной логики;
- Playwright для тач-взаимодействия и real pointer sequence.

In [ ]:
// Пример Playwright-скелета для регрессий клавиатуры
import { test, expect } from '@playwright/test';

test('keypad stays visible on rapid taps and gap taps', async ({ page }) => {
  await page.goto('http://localhost:5173');

  // Предусловие: открыть активный просчёт и сфокусировать поле факта.
  await page.getByPlaceholder('Фактическое количество').first().click();

  // Rapid taps по кнопкам.
  const key1 = page.getByRole('button', { name: '1' }).first();
  await key1.click();
  await key1.click();
  await key1.click();

  // Gap tap: координата внутри контейнера, но между кнопками.
  const keypad = page.locator('.fact-keypad').first();
  const box = await keypad.boundingBox();
  if (box) {
    await page.mouse.click(box.x + box.width * 0.5, box.y + box.height * 0.32);
  }

  await expect(page.locator('.fact-keypad')).toBeVisible();
});

test('expression input and delete still work', async ({ page }) => {
  await page.goto('http://localhost:5173');
  await page.getByPlaceholder('Фактическое количество').first().click();

  await page.getByRole('button', { name: '3' }).first().click();
  await page.getByRole('button', { name: '+' }).first().click();
  await page.getByRole('button', { name: '7' }).first().click();
  await page.getByRole('button', { name: '+' }).first().click();
  await page.getByRole('button', { name: '1' }).first().click();
  await page.getByRole('button', { name: '2' }).first().click();

  const input = page.getByPlaceholder('Фактическое количество').first();
  await expect(input).toHaveValue('3+7+12');

  await page.getByRole('button', { name: '⌫' }).first().click();
  await expect(input).toHaveValue('3+7+1');
});